# **03: Distillation of LLMs for text generation**

This notebook will delve into a crucial example: distilling language models. When distilling LLMs for text generation, the choice of probability divergence fundamentally changes the model's behavior. In such a task, the model must predict the next token from a vocabulary of over 30,000 words.

This notebook will follow the same path as the second one, just applied to a new task and maybe the most famous one today: text generation.

In [14]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from shrinkai.adapters import AttentionHeadSelector, FeatureExtractor
from shrinkai.distillation.distiller import Distiller
from shrinkai.distillation.losses import (
    AttentionMapLoss,
    HybridLoss,
    ProjectedFeatureLoss,
)

## **Adapting HuggingFace to match ShrinkAI**

See tutorial 02 for more details.

In [2]:
class CausalLMWrapper(FeatureExtractor):
    def __init__(self, model_name: str, layer_mapping: dict[int, str]):
        super(FeatureExtractor, self).__init__()
        self.hf_model = AutoModelForCausalLM.from_pretrained(
            model_name, attn_implementation="eager"
        )
        self.layer_mapping = layer_mapping
        self.target_layers = layer_mapping

    def forward(self, input_ids: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        attention_mask = (input_ids != self.hf_model.config.pad_token_id).long()

        outputs = self.hf_model(
            input_ids=input_ids, attention_mask=attention_mask, output_attentions=True
        )

        features_dict = {}
        if outputs.attentions is not None:
            for layer_idx, alias in self.layer_mapping.items():
                features_dict[alias] = outputs.attentions[layer_idx]

        return outputs.logits, features_dict

In [3]:
def build_shakespeare_dataloaders(batch_size: int = 8, max_length: int = 128):
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    dataset = load_dataset(
        "text",
        data_files="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    )
    dataset = dataset["train"].train_test_split(test_size=0.1)
    dataset["validation"] = dataset.pop("test")

    def tokenize_function(examples):
        return tokenizer(examples["text"])

    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

    def group_texts(examples):
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        total_length = (total_length // max_length) * max_length

        result = {
            k: [t[i : i + max_length] for i in range(0, total_length, max_length)]
            for k, t in concatenated_examples.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    lm_datasets = tokenized_datasets.map(group_texts, batched=True)
    lm_datasets.set_format(type="torch", columns=["input_ids", "labels"])

    train_dataset = TensorDataset(
        lm_datasets["train"][:]["input_ids"], lm_datasets["train"][:]["labels"]
    )
    val_dataset = TensorDataset(
        lm_datasets["validation"][:]["input_ids"],
        lm_datasets["validation"][:]["labels"],
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    sample_input = lm_datasets["validation"][:batch_size]["input_ids"]

    return train_loader, val_loader, sample_input, tokenizer.pad_token_id

In [4]:
train_loader, val_loader, sample_input, pad_token_id = build_shakespeare_dataloaders(
    batch_size=8, max_length=128
)

Map: 100%|██████████| 4000/4000 [00:00<00:00, 118336.08 examples/s]


In [5]:
# 12 layers
teacher = CausalLMWrapper("gpt2", {5: "attention_layer_1", 11: "attention_layer_2"})
teacher.hf_model.config.pad_token_id = pad_token_id

# 6 layers
student = CausalLMWrapper("distilgpt2", {2: "attention_layer_1", 5: "attention_layer_2"})
student.hf_model.config.pad_token_id = pad_token_id

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 9269.45it/s]


## **Reverse KL Loss**

In [6]:
from shrinkai.distillation.losses import ReverseKLLoss

Standard Knowledge Distillation uses **Forward KL Divergence** ($D_{KL}(P_{teacher} || P_{student})$). However, Forward KL is "mean-seeking". If the teacher thinks the next word could equally be "Sword" or "Shield", a Forward KL student will try to average these probabilities. This often leads to generating a safe, generic, or completely fabricated word that sits in the middle of the distribution, resulting in hallucinated or blurry text.

To fix this, modern LLM distillation uses **Reverse KL Divergence** ($D_{KL}(P_{student} || P_{teacher})$) combined with Causal Attention Distillation. Reverse KL is "mode-seeking". It heavily penalizes the student for predicting a word that the teacher considers impossible, but allows the student to confidently pick just one of the teacher's highly probable options and ignore the rest. As a consequence, the student generates sharp, highly coherent text, decisively committing to a specific grammatical path (e.g., choosing "Sword" and sticking with it) just like a real generative AI.

By combining `ReverseKLLoss` on the output vocabulary logits and `AttentionMapLoss` on the causal attention matrices, the student perfectly replicates the teacher's logical deduction steps without averaging out its creativity.

In [7]:
attention_loss = ProjectedFeatureLoss(
    projector=AttentionHeadSelector(heads_to_keep=list(range(12))),
    feature_loss=AttentionMapLoss(loss_type="mse"),
    project_teacher=True,
)

In [8]:
hybrid_loss = HybridLoss(
    logit_loss=ReverseKLLoss(temperature=4.0),
    feature_loss=attention_loss,
    feature_weight=10.0,
    convex_weighting=False,
)

In [9]:
distiller = Distiller(
    teacher=teacher,
    student=student,
    criterion=hybrid_loss,
    optimizer="adamw",
    lr=5e-5,
    weight_decay=0.01,
    device="auto",
)

In [10]:
distiller.feature_analysis(val_loader, metrics=["cka", "rsa", "attention"]).show()

          Representation Alignment Report (Feature Distillation)          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer / Stage Alias ┃ CKA (Linear) ┃ RSA (Pearson) ┃ Spatial Attention ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ attention_layer_1   │        0.994 │         0.185 │             0.911 │
│ attention_layer_2   │        0.996 │         0.760 │             0.986 │
└─────────────────────┴──────────────┴───────────────┴───────────────────┘

In [11]:
training_history = distiller.fit(train_loader, val_loader, epochs=3)

Epoch [01/03] Train Loss: 50.9165 - Train Acc: 2.89% | Val Loss: 35.6076 - Val Acc: 2.16%


Epoch [02/03] Train Loss: 43.1241 - Train Acc: 3.08% | Val Loss: 32.8491 - Val Acc: 2.16%


Epoch [03/03] Train Loss: 39.8957 - Train Acc: 3.08% | Val Loss: 31.2515 - Val Acc: 1.72%


In [12]:
distiller.benchmark(
    sample_input=sample_input,
    teacher_name="GPT 2",
    student_name="Tiny GPT 2",
    val_dataloader=val_loader,
).show()

                            Distillation Benchmark Report                             
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric            ┃ Teacher (GPT 2) ┃ Student (Tiny GPT 2) ┃    Gain / Compression ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ Parameters        │        124.44 M │              81.91 M │                -34.2% │
│ Model Size (Disk) │       474.75 MB │            312.50 MB │ -34.2% (1.5x smaller) │
│ Latency / Sample  │        26.91 ms │             14.73 ms │           1.8x faster │
│ Throughput (FPS)  │      37.2 img/s │           67.9 img/s │      +1.8x (67.9 FPS) │
│ Accuracy          │        2281.47% │             2118.10% │        92.8% retained │
└───────────────────┴─────────────────┴──────────────────────┴───────────────────────┘

In [13]:
distiller.feature_analysis(val_loader, metrics=["cka", "rsa", "attention"]).show()

          Representation Alignment Report (Feature Distillation)          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer / Stage Alias ┃ CKA (Linear) ┃ RSA (Pearson) ┃ Spatial Attention ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ attention_layer_1   │        0.994 │         0.211 │             0.936 │
│ attention_layer_2   │        0.997 │         0.792 │             0.990 │
└─────────────────────┴──────────────┴───────────────┴───────────────────┘

## **Handling Different Tokenizers and Vocabularies**

In this generative tutorial, both `gpt2` and `distilgpt2` belong to the same family and share the exact same tokenizer (vocabulary size of 50,257). This alignment allows us to directly apply the `ReverseKLLoss`. Because index `4256` represents the exact same token for both the teacher and the student, comparing their probability distributions is mathematically sound. 

But what if you want to distill a modern Llama-3 teacher (vocab size around 128k) into a smaller, custom architecture (vocab size around 32k)? 

When tokenizers differ, the output logit vectors have different dimensions, and more importantly, the indices no longer share the same semantic meaning. A direct probability divergence (like KL or Reverse KL) becomes meaningless. To bypass this barrier, you can adopt one of the following architectural shifts:

### **Pure Feature-Level Distillation**
Completely discard the logit-based loss. Instead of forcing the student to predict the same final vocabulary probabilities, force it to mimic the teacher's internal reasoning. By relying exclusively on `shrinkai`'s `AttentionMapLoss` and `ProjectedFeatureLoss`, the student learns the teacher's contextual geometry and spatial attention. You then apply a standard Cross-Entropy loss (using the ground-truth text tokenized by the *student's* tokenizer) to map those highly educated internal features to its own specific vocabulary.

### **Sequence-Level Knowledge Distillation**
Instead of aligning weights and probabilities during a single forward pass, you treat the teacher as a data generator. You prompt the massive teacher model to generate thousands of high-quality, specialized responses (e.g., generating text in the style of Shakespeare). Once this synthetic dataset is created, you train your student model on it using standard Causal Language Modeling with its own tokenizer. The distillation happens at the *sequence* level rather than the *tensor* level.

## **References**

1. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention Is All You Need*. NeurIPS.  
   arXiv:1706.03762  
   https://arxiv.org/abs/1706.03762

2. Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). *Language Models are Unsupervised Multitask Learners*. OpenAI.  
   arXiv:1902.09268  
   https://arxiv.org/abs/1902.09268

3. Hinton, G., Vinyals, O., & Dean, J. (2015). *Distilling the Knowledge in a Neural Network*. NIPS Deep Learning and Representation Learning Workshop.  
   arXiv:1503.02531  
   https://arxiv.org/abs/1503.02531

4. Gu, Y., Dong, L., Wei, F., & Huang, M. (2024). *MiniLLM: Knowledge Distillation of Large Language Models*. ICLR 2024.  
   arXiv:2306.08543  
   https://arxiv.org/abs/2306.08543

5. Wu, T., Tao, C., Wang, J., Yang, R., Zhao, Z., & Wong, N. (2025). *Rethinking Kullback-Leibler Divergence in Knowledge Distillation for Large Language Models*. COLING 2025, 5737–5755.  
   arXiv:2404.02657  
   https://arxiv.org/abs/2404.02657

6. Wang, W., Bao, H., Huang, S., Dong, L., & Wei, F. (2020). *MiniLM: Deep Self-Attention Distillation for Task-Agnostic Compression of Pre-Trained Transformers*. NeurIPS.  
   arXiv:2002.10957  
   https://arxiv.org/abs/2002.10957

7. Kim, Y., & Rush, A. M. (2016). *Sequence-Level Knowledge Distillation*. EMNLP 2016, 1317–1327.  
   arXiv:1606.07947  
   https://arxiv.org/abs/1606.07947

8. Romero, A., Ballas, N., Ebrahimi Kahou, S., Chassang, A., Gatta, C., & Bengio, Y. (2015). *FitNets: Hints for Thin Deep Nets*. ICLR.  
   arXiv:1412.6550  
   https://arxiv.org/abs/1412.6550

9. Loshchilov, I., & Hutter, F. (2019). *Decoupled Weight Decay Regularization*. ICLR.  
   arXiv:1711.05101  
   https://arxiv.org/abs/1711.05101

10. Kornblith, S., Norouzi, M., Lee, H., & Hinton, G. (2019). *Similarity of Neural Network Representations Revisited*. ICML, PMLR 97, 3519–3529.  
    arXiv:1905.00414  
    https://arxiv.org/abs/1905.00414

11. Kriegeskorte, N., Mur, M., & Bandettini, P. (2008). *Representational Similarity Analysis – Connecting the Branches of Systems Neuroscience*. Frontiers in Systems Neuroscience, 2, 4.  
    DOI: 10.3389/neuro.06.004.2008  
    https://doi.org/10.3389/neuro.06.004.2008

12. Sennrich, R., Haddow, B., & Birch, A. (2016). *Neural Machine Translation of Rare Words with Subword Units*. ACL.  
    arXiv:1508.07909  
    https://arxiv.org/abs/1508.07909